# 1. Train a sequence-to-function network
- **Input:** GB1 amino acid sequences and experimentally measured functional scores.
- **Model:** A learned amino acid embedding, two convolutional layers, and a regression head.
- **Training objective:** Minimize the mean squared error between predicted and measured scores.

Train a model, inspect its learning curves, and evaluate predictions on held-out variants.

## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the cells below.

In [ ]:
%pip -q install lightning==2.5.5
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import pandas as pd
import matplotlib.pyplot as plt

pl.seed_everything(7, workers=True)
torch.set_num_threads(4)

## 1. Prepare the data
GB1 scores reflect an IgG-binding selection assay. Each sequence differs from the reference by one or two substitutions. Encode amino acids as integer IDs; the model will learn their numerical representations.

In this dataset, `Q1A` replaces the glutamine immediately after the initial methionine. The conversion below maps each variant label to its full sequence.

In [ ]:
n_sequences = 12000

In [ ]:
source = 'https://raw.githubusercontent.com/RomeroLab/Seq2FxnNN/688d7bcf1455b9c7fd5fa7cda36089a340beca3d/gb1.tsv'
AA = 'ACDEFGHIKLMNPQRSTVWY'
WT = 'MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE'
aa_to_id = {aa: i for i, aa in enumerate(AA)}

def variant_sequence(variant):
    sequence = list(WT)
    for mutation in variant.split(','):
        sequence[int(mutation[1:-1])] = mutation[-1]
    return ''.join(sequence)

df = pd.read_csv(source, sep='\t').dropna(subset=['score']).drop_duplicates('variant')
df = df.sample(n=n_sequences, random_state=7).reset_index(drop=True)
df['sequence'] = df.variant.map(variant_sequence)
x = torch.tensor([[aa_to_id[aa] for aa in sequence] for sequence in df.sequence])
print(f'{len(df):,} sequences; token tensor: {tuple(x.shape)}')
display(df[['variant', 'sequence', 'score']].head())

## 2. Separate training, validation, and test data
Training data update the weights. Validation data select the best epoch. Test data assess the selected model. Standardize scores using the training set only.

This split evaluates predictions for new variants within the GB1 sequence landscape.

In [ ]:
batch_size = 128
training_fraction = 1.0
n_train, n_val = int(0.8 * len(x)), int(0.1 * len(x))
train_count = int(training_fraction * n_train)
y = torch.tensor(df.score.to_numpy(), dtype=torch.float32)
score_mean, score_std = y[:train_count].mean(), y[:train_count].std()
y_scaled = (y - score_mean) / score_std
train_loader = DataLoader(TensorDataset(x[:train_count], y_scaled[:train_count]), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(x[n_train:n_train+n_val], y_scaled[n_train:n_train+n_val]), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(x[n_train+n_val:], y_scaled[n_train+n_val:]), batch_size=batch_size)
print(f'Train: {train_count:,}; validation: {n_val:,}; test: {len(test_loader.dataset):,}')

## 3. Define the model and training steps
The embedding produces `[batch, length, channels]`. Convolutions use `[batch, channels, length]`; flattening preserves position-specific information for the regression head.

`training_step` returns a loss. Lightning performs backpropagation and optimizer updates. `validation_step` measures the same loss without updating weights.

In [ ]:
class SequenceRegressor(pl.LightningModule):
    def __init__(self, length, embedding_dim=8, channels=24, hidden_dim=64, learning_rate=0.003):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(20, embedding_dim)
        self.convolutions = nn.Sequential(nn.Conv1d(embedding_dim, channels, kernel_size=5, padding=2), nn.ReLU(), nn.Conv1d(channels, channels, kernel_size=5, padding=2), nn.ReLU())
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(length * channels, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))

    def forward(self, tokens):
        features = self.embedding(tokens).transpose(1, 2)
        return self.head(self.convolutions(features)).squeeze(-1)

    def shared_step(self, batch, stage):
        tokens, scores = batch
        loss = F.mse_loss(self(tokens), scores)
        self.log(stage + '_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, 'val')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

In [ ]:
epochs = 15
model = SequenceRegressor(length=x.shape[1], embedding_dim=8, channels=24, hidden_dim=64, learning_rate=0.003)
print(model)
print(f'{sum(p.numel() for p in model.parameters()):,} trainable parameters')

## 4. Train and inspect learning curves
Each epoch visits the training data once. The checkpoint with the lowest validation loss is used for evaluation. Compare training and validation curves to assess learning and overfitting.

In [ ]:
logger = CSVLogger('training_logs', name='protein_model')
checkpoint = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1)
trainer = pl.Trainer(max_epochs=epochs, accelerator='auto', devices=1, logger=logger, callbacks=[checkpoint], enable_model_summary=False, num_sanity_val_steps=0)
trainer.fit(model, train_loader, val_loader)
model = type(model).load_from_checkpoint(checkpoint.best_model_path).cpu().eval()

In [ ]:
metrics = pd.read_csv(logger.log_dir + '/metrics.csv')
for column in ['train_loss', 'val_loss']:
    values = metrics.dropna(subset=[column])
    plt.plot(values.epoch + 1, values[column], marker='o', label=column)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

## 5. Evaluate on held-out sequences
Convert predictions back to the original score units. Compare the network with a baseline that always predicts the training-set mean. Use the test set after choosing the model; use validation data for tuning.

In [ ]:
with torch.inference_mode():
    predictions = torch.cat([model(tokens) for tokens, scores in test_loader]) * score_std + score_mean
measured = y[n_train+n_val:]
mse = F.mse_loss(predictions, measured).item()
baseline_mse = ((measured - score_mean) ** 2).mean().item()
r_squared = 1 - ((predictions - measured) ** 2).sum().item() / ((measured - measured.mean()) ** 2).sum().item()
print(f'Test MSE: {mse:.3f}; mean-baseline MSE: {baseline_mse:.3f}; R²: {r_squared:.3f}')
plt.scatter(measured, predictions, s=8, alpha=0.3)
limits = [measured.min().item(), measured.max().item()]
plt.plot(limits, limits, color='black', linestyle='--')
plt.xlabel('Measured functional score'); plt.ylabel('Predicted functional score'); plt.show()

## Explore
Change one setting—`training_fraction`, embedding width, convolution channels, or learning rate—and retrain. Keep the data split fixed for model comparisons.

- How does model capacity affect training and validation loss?
- Which score ranges have the largest prediction errors?
- What would change if test variants were more distant from the training sequences?

## Save results
Save the selected model and its score normalization so predictions can be interpreted later.

In [ ]:
torch.save({'model_state': model.state_dict(), 'hyperparameters': dict(model.hparams), 'amino_acids': AA, 'reference_sequence': WT, 'score_mean': score_mean, 'score_std': score_std}, 'gb1_regressor.pt')
pd.DataFrame({'sequence': df.sequence.iloc[n_train+n_val:].to_numpy(), 'measured': measured.numpy(), 'predicted': predictions.numpy()}).to_csv('gb1_test_predictions.csv', index=False)

## References
[GB1 dataset](https://github.com/RomeroLab/Seq2FxnNN) · [LightningModule](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html) · [PyTorch modules](https://docs.pytorch.org/docs/stable/nn.html)